# E1--E4 manuscript plotting

This notebook reads the frozen `results/` artifacts and regenerates every
manuscript metric, density, and scatter figure. It never reruns a sampler.

Outputs are written in four publication formats, one directory per format:
`figures/png/` (600 dpi), `figures/tiff/` (600 dpi, LZW), `figures/svg/`, and
`figures/pdf/`. BAOAB is labelled **ULD** in all figures.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

HERE = Path.cwd().resolve()
ROOT = HERE.parent if HERE.name == "notebooks" else HERE
if not (ROOT / "src" / "manuscript.py").is_file():
    raise RuntimeError("Run this notebook from the repository's notebooks/ directory")
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".matplotlib-cache"))
sys.path.insert(0, str(ROOT))

from src.manuscript import EXPERIMENTS, METRICS, RESOURCE_AXES

print("Project root:", ROOT)
for key, spec in EXPERIMENTS.items():
    print(spec.number, key, "->", [spec.display_labels[m] for m in spec.methods])
print("Metrics:", METRICS)
print("Resource axes:", RESOURCE_AXES)

In [ ]:
from scripts.validate_release import validate_release

validate_release(ROOT, check_results=True, require_figures=False)
print("Frozen inputs are complete.")

In [ ]:
metric_command = [
    sys.executable,
    str(ROOT / "scripts" / "replot_manuscript_figures.py"),
    "--results-dir", str(ROOT / "results"),
    "--figures-dir", str(ROOT / "figures"),
    "--no-clean",
]
print("Running:", " ".join(metric_command))
subprocess.run(metric_command, cwd=ROOT, env=os.environ.copy(), check=True)

In [ ]:
sample_command = [
    sys.executable,
    str(ROOT / "scripts" / "replot_generated_samples.py"),
    "--results-root", str(ROOT / "results"),
    "--output-root", str(ROOT / "figures"),
    "--cache-root", str(ROOT / "cache" / "generated_samples"),
    "--manifest-path",
    str(ROOT / "cache" / "generated_samples" / "generated_sample_plots_manifest.json"),
    "--overwrite",
]
print("Running:", " ".join(sample_command))
subprocess.run(sample_command, cwd=ROOT, env=os.environ.copy(), check=True)

In [ ]:
report = validate_release(
    ROOT,
    check_results=True,
    require_figures=True,
)
from src.manuscript import FIGURE_FORMATS

print("Final validation:", report["status"])
for _ext in FIGURE_FORMATS:
    _files = sorted((ROOT / "figures" / _ext).glob(f"*.{_ext}"))
    print(f"figures/{_ext}: {len(_files)} files")